# §4 体验实验：MLP 全推 + 梯度检查

这个笔记本只用一个微型神经网络（输入 3 个数 → 隐藏 4 个神经元 → 输出 2 类，**共 26 个参数**），亲手做两件全系列最重要的事：**前向传播**（输入一路算出预测）和**反向传播**（把误差沿来路传回去，算出每个参数的责任），然后用笨办法逐一验证手推公式没推错。

- **网络**：3→4→2，隐藏层用 tanh（把任意实数压进 −1 到 1 的激活函数），输出层用 softmax（把输出变成和为 1 的概率），损失用交叉熵（只看正确类别的概率，取负对数）。
- **验证方法**：反向传播给出的**解析梯度**（手推公式的答案），和**数值梯度**（把每个参数轻轻拨动一下、直接量损失变化的笨办法）逐一对账，26 个参数的相对误差全部小于 1/100000 才算 PASS。
- **安装说明**：Colab 已预装 NumPy，直接运行即可，无需 pip 单元格。

本笔记本中的代码与输出均来自 2026-10-08 的真实运行记录。

## 1. 搭网络：初始化

神经网络的全部家当就是两组权重和两组偏置：权重是每条连线上的放大器，偏置是每个神经元的起手值。固定随机种子，任何人重跑这格，数字都一模一样。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)   # 固定种子：任何人重跑，数字逐字相同

# 网络：输入 3 维 → 隐藏 4 个神经元（tanh）→ 输出 2 类（softmax）
# 参数共 26 个 = W1(3×4=12) + b1(4) + W2(4×2=8) + b2(2)
W1 = rng.normal(0, 0.5, (3, 4))  # 第一层权重：每条"输入→隐藏"连线一个数
b1 = np.zeros(4)                 # 第一层偏置：每个隐藏神经元一个"起手值"
W2 = rng.normal(0, 0.5, (4, 2))  # 第二层权重：每条"隐藏→输出"连线一个数
b2 = np.zeros(2)                 # 第二层偏置

x = np.array([0.5, -0.3, 0.8])   # 输入样本：3 个数
y = np.array([0.0, 1.0])         # 真实标签：one-hot，意思是"正确答案是第 2 类"
print(f'参数量: W1 {W1.size} + b1 {b1.size} + W2 {W2.size} + b2 {b2.size} = '
      f'{W1.size + b1.size + W2.size + b2.size} 个')

## 2. 前向传播

前向传播就是数据的流水线：每个神经元把输入各自乘权重再相加（加权求和），加偏置，最后过一道激活函数。输出层再过 softmax，把两个数变成和为 1 的概率。

In [ ]:
def forward(x, W1, b1, W2, b2):
    z1 = x @ W1 + b1             # 第一步：每个隐藏神经元 = 3 个输入各乘权重再相加，加偏置
    a1 = np.tanh(z1)             # 第二步：tanh 激活，把任意实数压进 (-1,1)，引入非线性
    z2 = a1 @ W2 + b2            # 第三步：输出层同样的加权求和
    e = np.exp(z2 - z2.max())    # 第四步：softmax，先指数化（减去最大防爆数）
    p = e / e.sum()              #        再除以总和，两个输出变成和为 1 的概率
    return z1, a1, z2, p

z1, a1, z2, p = forward(x, W1, b1, W2, b2)
print(f'前向结果: 输出概率 p = [{p[0]:.6f}, {p[1]:.6f}]（和={p.sum():.6f}）')

## 3. 损失：预测错得有多离谱

损失用**交叉熵**：只盯着正确类别那个概率，取负对数——概率越接近 1，损失越接近 0；概率越小，损失越大得离谱。这是训练时要压低的那座山的高度读数。

In [ ]:
def loss(p, y):
    return -np.log(p[y.argmax()])  # 交叉熵：只看正确类别那个概率，取负对数

L = loss(p, y)
print(f'损失 L = -ln(p[正确类别]) = {L:.6f}')

## 4. 逐步反向传播

反向传播用**链式法则**（复合函数求导规则：总变化率 = 每一环的变化率连乘），把损失沿原路一步步传回去，算出每个参数的**梯度**——即动这个参数一小下，损失会变多少。下面七行代码和公式推导一一对应，逐行看：

In [ ]:
# 公式推导顺序：L → z2 → (W2,b2) 和 a1 → z1 → (W1,b1)
dz2 = p - y                      # ① softmax+交叉熵合起来的导数：预测概率 − 真实标签
dW2 = np.outer(a1, dz2)          # ② L 对 W2 的梯度 = 上游信号 dz2 × 本层输入 a1（外积：列×行）
db2 = dz2.copy()                 # ③ L 对 b2 的梯度 = 上游信号（偏置的输入恒为 1）
da1 = W2 @ dz2                   # ④ 误差沿连线回流：L 对隐藏层输出 a1 的梯度
dz1 = da1 * (1 - a1 ** 2)        # ⑤ 穿过 tanh：乘它的导数 1−a1²（a1 就是 tanh 的输出，直接可用）
dW1 = np.outer(x, dz1)           # ⑥ L 对 W1 的梯度 = 上游信号 dz1 × 本层输入 x
db1 = dz1.copy()                 # ⑦ L 对 b1 的梯度 = 上游信号

grads = {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

## 5. 数值梯度：用笨办法对答案

怎么知道手推的七行没推错？用**有限差分法**这个笨办法：把某个参数先加一点点、再减一点点，各算一次损失，损失的差除以参数的差，就是该参数处斜率的直接测量值。完全不靠任何求导公式。对 26 个参数各量一遍：

In [ ]:
EPS = 1e-5                       # 拨动的大小：够小才准，又不能小到被浮点误差淹没
num_grads = {}
for name, W in [('W1', W1), ('b1', b1), ('W2', W2), ('b2', b2)]:
    g = np.zeros_like(W)
    it = np.nditer(W, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = W[idx]
        W[idx] = old + EPS
        lp = loss(forward(x, W1, b1, W2, b2)[3], y)   # θ+ε 的损失
        W[idx] = old - EPS
        lm = loss(forward(x, W1, b1, W2, b2)[3], y)   # θ−ε 的损失
        W[idx] = old                                  # 量完必须拨回来
        g[idx] = (lp - lm) / (2 * EPS)                # 斜率 = 损失变化量 / 参数变化量
        it.iternext()
    num_grads[name] = g

## 6. 逐参数对账与判定

相对误差 = 两种答案之差的绝对值 ÷ 两个绝对值之和。分母加上两者之和，是为了防止两边都接近 0 时读数爆炸。26 个参数全部小于 1/100000（1e-5）才算 PASS：

In [ ]:
print('逐参数对账表（解析梯度 vs 数值梯度）')
print(f'{"参数":<8}{"解析梯度":>14}{"数值梯度":>14}{"相对误差":>13}')
print('-' * 51)
max_err = 0.0
for name in ['W1', 'b1', 'W2', 'b2']:
    for idx, _ in np.ndenumerate(grads[name]):
        a, n = grads[name][idx], num_grads[name][idx]
        err = abs(a - n) / (abs(a) + abs(n) + 1e-30)
        max_err = max(max_err, err)
        print(f'{name}{idx!s:<5}{a:>14.8f}{n:>14.8f}{err:>13.3e}')
print('-' * 51)
print(f'26 个参数中最大相对误差: {max_err:.3e}')
print(f'判定: 全部 < 1e-5 → {"PASS" if max_err < 1e-5 else "FAIL"}')

## 7. 观察要点

- **26 个参数全部对账通过，最大相对误差 4.6×10⁻¹¹**——比 1e-5 的合格线小了 6 个数量级。手推的链式法则七行，和计算机暴力测量的斜率，前 8 位小数完全一致：反向传播公式是对的。
- **b2 的两个数互为相反数（0.593 与 −0.593）**：这是 softmax 两概率之和恒为 1 留下的结构性痕迹——正确类别的概率涨多少，错误类别就跌多少，dz2 = p − y 的两个分量之和必为 0。
- **解析梯度与数值梯度逐位对照的意义**：神经网络训练就是把每个参数往梯度反方向挪一点；如果梯度本身算错了，后面所有步数都是在错误的方向上认真前进。这一步是全部训练的信任地基。
- **为什么数值梯度只做对账、不能拿来训练**：26 个参数要量 52 次前向；真网络上亿参数就要量两亿次——笨办法贵到用不了，但它慢得诚实，所以能当裁判。
- 实测打印原文、机器环境与判定结果见同目录《run_log.txt》。